# CH 6 학습 관련 기술
신경망 학습의 핵심 개념을 배우는데 갇중치 매개 변수의 최적값, 등을 배워보자.

## 6.1 매개변수 갱신
신경망 학습의 목적은 손실 함수의 값을 가능한 한 낮추는 매개변수를 찾는 것이다. 즉 매개변수의 최적값을 찾는 문제이고 이를 최적화(Optimization) 라고 한다. 매개변수 공간은 매우 넓고 복잡하다. 심지어 심층 신경망은 매개변수가 많아서 더 어렵다. 우리는 지금까지 매개변수의 기울기를 이용해서 기울어진 방향으로 값을 갱신하는 일을 반복하여 최적 값을 찾아 나갔는데 이를 확률적 경사 하강법(SGD)라고 한다. 하지만 이 보다 더 나은 방법을 알아보자.

### SGD
기울어진 방향으로 일정 거리만 가겠다는 방법이다.

단점이 있는데 단순하고 구현도 쉽지만 문제에 따라서 비효율적일 때가 있다. 비등방성 함수(방향에 따라 기울기가 달라지는 함수)에서는 탐색 경로가 비효율적이다. 

In [ ]:
class SGD:
    def __init__(self, lr = .01): # lr은 학습률을 의미한다.
        self.lr = lr

    def update(self, params, grads):
        for key in params.keys():
            params[key] -= self.lr * grads[key] # grads[key]는 W에 대한 손실함수의 기울기이다.

### Momentum
기울기 방향으로 힘을 받아 물체가 가속 되는 것처럼 이동 시키는 것이다.
아래의 식을 바라보면 v를 딕셔너리 형태로 가지고 있다. 그러다가 초기의 기울기를 받고 이를 새로운 기울기로 살짝씩만 이동하는 경향을 바꾸어 이동한다.
공이 바닥을 구르듯 움직인다. SGD에 비하면 지그재그한 정도가 덜하다.(그 이유는?) 더 자세히 말하면 x축의 힘은 아주 작지만 방향은 변하지 않아서 한 방향으로 일정하게 가속되지만 y축의 힘은 크지만 위아래로 번갈아 받아서 상충하여 속도가 안정적이지 않다.(이건 책의 그래프를 기반으로 설명하는 것인데 왜 그런걸까? 일반적으로 그런 것은 아니다.)

In [ ]:
class Momentum:
    def __init__(self, lr=.01, momentum=.9):
        self.lr = lr
        self.momentum = momentum
        self.v = None
    
    def update(self, params, grads):
        if self.v is None:
            self.v = {}
            for key, val in params.items():
                self.v[key] = np.zeros_like(val)
            
        for key in params.keys():
            self.v[key] = self.momentum*self.v[key] - self.lr*grads[key]

### AdaGrad
학습률 lr의 값이 신경망 학습에서는 중요하다. 이 값과 학습시간은 반비례하여서 잘 정해야 한다. 이를 정하는 효과적인 기술로 학습률 감소가 있다. 학습을 진행하면서 학습률을 점차 줄이는 것이다. 이를 낮추는 간단한 방법은 매개변수 전체의 학습률 값을 일괄적으로 낮추는 것이다. 이를 발전 시킨 것이 AdaGrad이다. AdaGrad는 각각의 매개변수에 맞춤형 값을 만든다. 개별 매개변수에 적응적으로 학습률을 조정하면서 학습을 진행한다.  
h라는 변수를 추가한다. 행렬 간의 원소 간의 곱셈을 한 값을 기존 h 값에 더하여 h값의 갱신이 이루어진다. 1/root(h)를 기울기와 학습률에 곱해주어 학습률을 낮아지게 한다. 즉, 학습률 감소가 매개변수의 원소마다 다르게 적용되게 된다.  
무한히 학습하면 0이 되게 되는데 이를 개선한 기법이 RMSProp이라고 있다.

In [ ]:
class AdaGrad:
    def __init__(self, lr=.01):
        self.lr = lr
        self.h = None
    
    def update(self, params, grads):
        if self.h is None:
            self.h = {}
        
            for key, val in params.items():
                self.h[key] = np.zeros_like(val)
        
        for key in params.keys():
            self.h[key] += grads[key] * grads[key]
            params[key] -= self.lr * grads[key] / (np.sqrt(self.h[key]) + 1e-7) # 1e-7은 0이 담겨 있다고 해도 0으로 나누는 사고를 막아준다.

### Adam
https://arxiv.org/pdf/1412.6980v8

Adam은 간단히 말해 **"방향은 Momentum처럼(관성), 보폭은 AdaGrad처럼(개별 조절)"** 하는 방식입니다.
① 첫 번째 기록: 방향(Momentum)
이전의 흐름을 기억하여 가속도를 붙입니다.
$$m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla L(W_t)$$
* $m_t$: 기울기의 **지수 이동 평균** (1차 모멘트)
* $\beta_1$: 보통 $0.9$로 설정 (과거 방향을 $90\%$ 반영)

② 두 번째 기록: 보폭(Adaptive Learning Rate)
기울기가 변화하는 크기를 기억하여 매개변수마다 보폭을 다르게 조절합니다.
$$v_t = \beta_2 v_{t-1} + (1 - \beta_2) (\nabla L(W_t))^2$$
* $v_t$: 기울기 제곱의 **지수 이동 평균** (2차 모멘트)
* $\beta_2$: 보통 $0.999$로 설정 (기울기의 진동 폭을 아주 길게 기억)

③ 편향 수정 (Bias Correction)
학습 초기에는 $m_t$와 $v_t$가 $0$으로 편향되는 경향이 있어 이를 보정해줍니다.
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$

④ 최종 가중치 업데이트
최종적으로 가중치($W$)를 다음과 같이 업데이트합니다.
$$W_{t+1} = W_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$
* **분자($\hat{m}_t$):** "가던 방향으로 가자!" (모멘텀)
* **분모($\sqrt{\hat{v}_t}$):** "많이 변한 애는 조금만, 적게 변한 애는 크게 움직이자!" (어댑티브 학습률)

그런데 지수 이동 평균인 이유는 이 과정을 반복해서 이루어지기 때문이다.

### AdamW
Regularization은 훈련데이터가 오퍼피팅 되는 것을 막아서 성능이 잘 안 나오게 막는다. 그 과정을 손실함수의 결과 가중치 W 값을 더하는 방식으로 이루어진다. 하지만 이 상태로 optimazation의 방법 중 하나인 Adam에 넣게 되면 가중치 W 값도 보폭 조절에 휘말리게 된다. 즉, 가중치 감쇠(Weight Decay) 적용: * $W_{next} = W - \eta \Delta W \mathbf{ - \eta \lambda W}$마지막 단계에서 현재 가중치 값의 일정 비율($\eta\lambda$)만큼을 그냥 깎아버립니다. 하지만 이렇게 된다면 가중치 값도 변형되게 된다. 이런 점을 해결하기 위해서 Optimazation이 이루어진 이후에 적용하기로 했다. 그 방법이 바로 AdamW이다.  
Adam과 수식적으로 차이는 없지만 가중치의 순서가 다른 것이다.

In [ ]:
class Adam:
    def __init__(self, lr=.001, beta1=.9, beta2=.999):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.iter = 0
        self.m = None
        self.v = None

    def update(self, params, grads):
        if self.m if None:
            self.m, self.v = {}, {}
            for key, val in params.items():
                self.m[key] = np.zeros_like(val)
                self.v[key] = np.zeros_like(val)
        
        self.iter += 1
        lr_t = self.lr * np.sqrt(1.0 - self.beta28*self.iter) / (1.0 - self.beta1**self.iter)

        for key in params.keys():
            self.m[key] += (1 - self.beta1) * (grads[key] - self.m[key]) # Momentum (방향)
            self.v[key] += (1 - self.beta2) * (grads[key] ** 2 - self.v[key]) # 보폭

            params[key] -= lr_t * self.m[key] / (np.sqrt(self.v[key]) + 1e-7)

## 6.2 가중치의 초깃값
오버 피팅을 억제해서 범용 성능을 높이는 테크닉인 가중치 감소 기법을 소개한다.  

### 초깃값을 0으로 하면?
가중치 감소는 가중치 매개변수의 값이 작아지도록 즉 오버피팅을 일어나지 않게 학습하는 방법이다. 그러면 가중치의 초깃값을 아예 작은 값 0으로 하면 되지 않으까? 그렇게 되면 오차역전파법에서 모든 가중치의 값이 똑같이 갱신되므로 학습이 올바로 이뤄지지 않는다. 순전파 일 때도 가중치가 0이므로 뉴런에 모두 같은 값이 전달된다. 편향 값만 남기 때문이다. 그래서 초깃값은 무작위로 설정해야 한다.

### 은닉층의 활성화값 분포
가중치 초깃값에 따라 은닉층의 활성화값(활성화 함수의 출력 데이터)의 분포를 살펴보자.  
가중치 초깃값을 표준 편차(데이터가 모여있는 정도를 나타내는데 크면 흩어져 있고 작으면 모여 있다.)가 1인 정규분포로 하고 시그모이드 함수를 사용할 때는 0과 1에 치우쳐져 분포하게 되게 되는데 이렇게 되면 역전파의 기울기 값이 점점 작아지다가 사라진다. 이를 기울기 소실 문제라고 한다. 표준편차를 .01로 바꾸어 보면 .5부근에서 집중되게 되면서 기울기 소실 문제가 이루어지지 않는다. 하지만 이 경우에는 다수의 뉴런들이 거의 같은 값을 출력하게 되어 뉴런을 여러 개 둔 이유가 사라지게 되어 표현력을 제한하게 된다.  
그래서 Xavier 초깃값을 사용하기도 한다. 앞 계층의 노드가 n 개 일 때 표준편차를 1/root(n)으로 하면 적절한 분포를 얻을 수 있다는 것이다.  
하지만 Xavier 초깃값은 활성화 함수가 선형적인 것을 전제로 이끈 것인데 시그모이드 함수는 좌우대칭적이여서 중앙 부근이 선형적인 함수로 볼 수 있다. 그래서 ReLU에 특화된 초깃값을 사용해야 하는데 이를 He 초깃값, 카이밍 초깃값이라고 한다. 앞 계층의 노드가 n 개 일 때 표준편차를 root(2/n)으로 하면 적절한 분포를 얻을 수 있다는 것이다. Xavier 와 root(2)차이가 나는데 ReLU 함수가 음의 영역에서 0잉여서 더 넓게 분포 시키기 위해서 그런 것이라 해석할 수 있다.  
Xavier 초깃값은 층이 깊어지면서 치우침이 생기면서 기울기 소실 문제를 일으키게 되지만 He 초깃값은 그렇지 않다.

## 배치 정규화 Batch Normalization
각 층이 활성화를 적당히 퍼뜨리도록 강제 해보자

### 배치 정규화 알고리즘
배치 정규화 알고리즘을 통해서 학습을 빨리 진행 시킬 수 있고 초깃값에 의존하지 않고 오버피팅을 억제할 수 있다. 이 배치 정규화는 층과 층 사이에서 활성화 함수가 적용되기 전에 시행 되는 알고리즘이다. 방식은 데이터의 분포를 평균이 0 그리고 분산이 1이 되도록 정규화 한다. 그리고 정규화한 데이터에 고유한 확대와 이동 변환을 수행한다.

## 바른 학습을 위해서
훈련데이터에만 지나치게 적응되어 그 이외의 데이터에 제대로 대응하지 못하는 상태를 오버피팅이라고 한다.

### 오버피팅
오버피팅은 매개변수가 많고 표현력이 많은 모델과 훈련 데이터가 적을 때 일어난다. overfit_weight_decay.py 참고

### 가중치 감소
오버피팅 억제 용으로 가중치 감소라는 것이 있다. 학습 과정에서 큰 가중치에 대해서 그에 상응하는 큰 페널티를 부과하여 오버피팅을 억제하는 방법이다. 왜 이런 방식을 사용하냐면 오버피팅은 가중치 매개변수의 값이 커서 발생하는 경우가 많다.  
신경망 학습은 손실 함수의 값을 줄이기 위한 것인데 이때 자중치의 제곱 노름(L2 노름)을 손실함수에 더한다. 그러면 가중치가 커지는 것을 억제할 수 있다. 그래서 $L_{total} = L_{data} + \frac{\lambda}{2} \|W\|^2$를 손실함수에 더한다. 람다는 여기서 하이퍼 파라미터이다. 그래서 가중치 감소는 모든 가중치 각각의 손실함수에 $L_{total} = L_{data} + \frac{\lambda}{2} \|W\|^2$를 더한다. 따라서 가중치의 기울기를 구하는 계산에서는 그동안의 오차역전팝법에 따른 결과에 정규화한 항을 미분한 $\lambda R(W)$를 더한다.

### 드롭아웃
신경망 모델이 복잡해지면 가중치 감소만으로 대응하기 어려워진다. 이럴 때 드롭아웃이라는 기법을 사용한다. 이 기법은 훈련시에 은닉층의 뉴런을 임의로 삭제하면서 학습하는 방법이다. 그리고 테스트 때에는 삭제 하지 않은 채로 출력한다.                  

In [ ]:
class Dropout:
    def __init__(self, dropout_ratio=.5):
        self.dropout_ratio = dropout_ratio
        self.mask = None

    def forward(self, x, train_flag=True):
        if train_flag:
            self.mask = np.random.rand(*x.shape) > self.dropout_ratio
            return x * self.mask
        else:
            return x * (1.0 - self.dropout_ratio)

    def backward(self, dout):
        return dout * self.mask

## 적절한 하이퍼파라미터 값 찾기
각 층의 뉴런 수, 배치 크기, 매개변수 갱신시의 학습률, 가중치 감소... 이러한 하이퍼파라미터의 값을 적절히 설정하지 않으면 모델 성능이 크게 떨어진다. 최대한 효율적으로 탐색하는 방법을 알아보자.

### 검증 데이터
하이퍼파라미터의 성능을 평가할 때는 테스트 데이터를 사용해서는 안된다. 그런 이유는 테스트 데이터를 이용해서 조정하면 값이 테스트 데이터에 오버피팅된다. 그래서 하이퍼파라미터 전용 확인 데이터가 필요하다. 이를 일반적으로 검증 데이터라고 한다. 일반적으로 훈련 데이터에 20% 정도 분리한다.

### 하이퍼 파라미터 최적화
하이퍼파라미터 최적화의 핵심은 최적 값이 존재하는 범위를 조금씩 줄여나가는 것이다. 그러려면 먼저 범위를 설정하고 무작위로 값을 골라서 그 값으로 정확도를 평가한다. 그리고 이 과정을 반복하여 범위를 좁힌다.  
범위는 대략적으로 지정하는 것이 효과적이고 실제로 10의 거듭제곱 단위로 범위를 지정한다. (로그 스케일)  

실제 구현은 hyperparameter_optimization.py를 참고하자.